In [32]:
# ============================================================
# 022_research_frontier_gap_mining
# ============================================================
#
# Overview
# ----------------
# This notebook extracts, prioritizes, and maintains an updatable, cluster-wise
# list of "unresolved research gaps" from citation network analysis.
# It is designed for recurring (e.g., weekly) research operations, supporting
# systematic identification, evaluation, and tracking of research opportunities
# over time.
#
# The notebook treats research gaps as evolving objects rather than static outputs,
# using Notion as a persistent state store and reference point for temporal diffs.
#
#
# Inputs / Outputs
# ----------------
# **Inputs (conceptual):**
# - Graph / centrality artifacts from Day21
#   (paper nodes with citation-based metrics and community/cluster assignments)
# - Representative papers per cluster
# - Optional citation edges
# - Structured paper summaries (abstracts, limitations, future work), if available
# - Notion "Research Gaps" database
#   (serves as the persistent store for previous runs and diff computation)
#
# **Outputs:**
# 1. Aggregated limitations / future work per cluster
# 2. Cluster-wise research gaps, categorized as:
#    - coverage gaps (unexplored areas or missing integrations)
#    - measurement gaps (data or metric limitations)
#    - causal inference gaps (identification challenges)
# 3. (Private) gap extraction prompt template
# 4. (Private) gap → candidate data source mapping
# 5. Gap prioritization and ranking scores
# 6. Cluster-wise gap dashboards (Markdown / Notion-ready)
# 7. Notion export:
#    - Upsert into the "Research Gaps" database (key = gap_id)
# 8. Diff report vs previous run:
#    - New gaps
#    - Missing gaps
#    - Significant score changes (computed against Notion state)
#
#
# Structure
# ----------------
# Cell 00 — Purpose / Today's Goal / Outputs
# Cell 01 — Dependencies & Config (CFG)
# Cell 02 — Load Inputs (Network & Summaries)
# Cell 03 — Select Representative Papers per Cluster
# Cell 04 — Build Evidence Packs for Gap Extraction
# Cell 05 — (Private Hook) Gap Extraction Prompt Template
# Cell 06 — Cluster-wise Gap Extraction (LLM)
# Cell 07 — Gap Normalization
# Cell 08 — Gap Prioritization
# Cell 09 — (Private Hook) Gap → Data Mapping
# Cell 10 — Cluster-wise Gap Dashboard
# Cell 11 — Notion Export (Upsert Research Gaps DB)
# Cell 12 — Diff vs Previous Run (Notion-based)
#
#
# Notes
# ----------------
# - This notebook follows researchOS artifact-based conventions.
# - Uses OpenAI API (gpt-4o-mini) only; no Claude/Anthropic SDK.
# - Secrets (API keys, Notion IDs) are loaded from env.txt.
# - Designed to be runnable end-to-end with "Run All".
# - Notion is treated as the single source of truth for previous-run state.
# - The Notion "Research Gaps" database uses its Title property (e.g., "Name")
#   as the unique gap identifier (= gap_id).
# - Schema drift across artifacts (e.g., cluster_id vs community_id) is handled
#   defensively within the pipeline.
# - Graceful fallbacks are implemented for missing optional inputs.
#
# ============================================================

In [22]:
# ============================================================
# Cell 01 — Dependencies & Config (CFG)
# ============================================================
# Overview:
# - Import required libraries
# - Load environment variables from env.txt
# - Configure LLM settings (OpenAI gpt-4o-mini)
# - Set execution mode (dry_run) and input resolution strategy
#
# Inputs / Outputs:
# - Inputs: env.txt (environment variables)
# - Outputs: CFG object with all configuration
#
# Notes:
# - API keys must be in env.txt (OPENAI_API_KEY)
# - dry_run=True skips LLM calls and creates placeholders
# - input_dir_mode controls how we resolve upstream artifacts

import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
from typing import Dict, List, Optional, Tuple
import warnings
from dotenv import load_dotenv
from openai import OpenAI
from difflib import SequenceMatcher
import hashlib

# ------------------------------------------------------------
# Load environment variables
# ------------------------------------------------------------
# Explicitly load env.txt (instead of default .env)
load_dotenv("env.txt")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
class CFG:
    # LLM Configuration
    llm_provider = "OpenAI"
    llm_model = "gpt-4o-mini"
    llm_temperature = 0.0
    
    # Execution mode
    dry_run = False  # Set to True to skip LLM calls
    
    # Input resolution
    input_dir_mode = "latest_day21_paper_graph"  # or "explicit_path"
    input_dir = None  # Used only if input_dir_mode == "explicit_path"
    
    # Day21 artifacts root
    day21_artifacts_root = "./artifacts"
    day21_run_dir_prefix = "day21_paper_graph"  
    
    # Optional Day20 enrichment (recommended)
    enable_day20_enrichment = True
    day20_artifacts_root = "./artifacts/day20"
    day20_run_dir_prefix = None  # run dir is YYYYMMDD_HHMMSS so None
    day20_enriched_pool_filename = "candidate_pool_enriched_scored.csv"
    
    # Representative papers per cluster
    k_representatives = 10
    
    # Gap prioritization weights
    priority_weights = {
        "importance": 0.5,
        "feasibility": 0.3,
        "novelty": 0.2
    }
    
    # Gap normalization
    similarity_threshold = 0.85  # For merging near-duplicates
    
    # Output directory
    output_dir = f"./artifacts/day22_gap_mining/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # Previous run for diff (optional)
    previous_gaps_path = None  # Set to compare with previous run
    
    # Prompt paths (private hooks)
    prompt_gap_extract_path = "./prompts/private/gap_extraction.txt"
    prompt_data_map_path = "./prompts/private/gap_to_data.txt"
    
    # Drive/Notion export (optional)
    enable_drive_export = False
    enable_notion_export = True
    drive_root = "./drive/gaps"
    notion_database_id = None  # Set if using Notion

# Create output directory
os.makedirs(CFG.output_dir, exist_ok=True)



print(f"Configuration loaded")
print(f"- LLM: {CFG.llm_provider} / {CFG.llm_model}")
print(f"- Dry run: {CFG.dry_run}")
print(f"- Input mode: {CFG.input_dir_mode}")
print(f"- Output dir: {CFG.output_dir}")

Configuration loaded
- LLM: OpenAI / gpt-4o-mini
- Dry run: False
- Input mode: latest_day21_paper_graph
- Output dir: ./artifacts/day22_gap_mining/20260118_145443


In [3]:
# ============================================================
# Cell 02 — Load Inputs (Network & Summaries)
# ============================================================
# Overview:
# - Resolve INPUT_DIR from CFG (latest Day21 run or explicit path)
# - Load canonical graph inputs by semantic role (not hardcoded filenames)
# - Load nodes_with_metrics.csv, top_papers_by_community.csv
# - Optional: edges.csv, summaries
# - Graceful fallbacks for missing optional data
#
# Inputs / Outputs:
# - Inputs: Day21 artifacts directory
# - Outputs: nodes_df, clusters_df, edges_df (optional), summaries_df (optional)
#
# Notes:
# - Do NOT require nodes.csv or clusters.csv by name
# - Use best-available inputs in INPUT_DIR
# - Joins by stable identifier (openalex_id/openalex_wid/paper_id)

# ------------------------------------------------------------
# Helper: Check if directory name matches run format
#   - supports:
#       YYYYMMDD_HHMMSS
#       <prefix>_YYYYMMDD_HHMMSS
# ------------------------------------------------------------
def _extract_run_timestamp(dirname: str, prefix: Optional[str] = None) -> Optional[str]:
    """
    Return timestamp string 'YYYYMMDD_HHMMSS' if dirname matches:
      - 'YYYYMMDD_HHMMSS'
      - '<prefix>_YYYYMMDD_HHMMSS'  (when prefix is provided)
    Otherwise return None.
    """
    s = dirname.strip()

    # Case A: pure timestamp
    if re.fullmatch(r"\d{8}_\d{6}", s):
        return s

    # Case B: prefixed timestamp
    if prefix:
        pat = rf"^{re.escape(prefix)}_(\d{{8}}_\d{{6}})$"
        m = re.match(pat, s)
        if m:
            return m.group(1)

    return None


def find_latest_run_dir(root: str, prefix: Optional[str] = None) -> Optional[str]:
    """
    Find the latest run directory under root.
    If prefix is provided, only consider directories whose name is:
      '<prefix>_YYYYMMDD_HHMMSS'
    Also supports directories named exactly 'YYYYMMDD_HHMMSS' when prefix is None.
    """
    root_path = Path(root)
    if not root_path.exists():
        return None

    candidates = []
    for d in root_path.iterdir():
        if not d.is_dir():
            continue
        ts = _extract_run_timestamp(d.name, prefix=prefix)
        if ts:
            candidates.append((ts, d))

    if not candidates:
        return None

    # Sort by timestamp ascending, pick latest
    candidates.sort(key=lambda x: x[0])
    return str(candidates[-1][1])

# ------------------------------------------------------------
# Helper: Extract OpenAlex WID from URL or WID
# ------------------------------------------------------------
def to_openalex_wid(x) -> Optional[str]:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    if not s:
        return None
    if s.startswith("W"):
        return s
    if "/W" in s:
        return "W" + s.split("/W")[-1]
    return None


# ------------------------------------------------------------
# Resolve INPUT_DIR
# ------------------------------------------------------------
if CFG.input_dir_mode == "latest_day21_paper_graph":
    INPUT_DIR = find_latest_run_dir(CFG.day21_artifacts_root, prefix=CFG.day21_run_dir_prefix)
    if INPUT_DIR is None:
        raise FileNotFoundError(
            f"No run directories found under {CFG.day21_artifacts_root}"
        )
    print(f"Resolved INPUT_DIR (latest): {INPUT_DIR}")
elif CFG.input_dir_mode == "explicit_path":
    INPUT_DIR = CFG.input_dir
    if INPUT_DIR is None:
        raise ValueError("input_dir must be set when input_dir_mode='explicit_path'")
    print(f"Using explicit INPUT_DIR: {INPUT_DIR}")
else:
    raise ValueError(f"Unknown input_dir_mode: {CFG.input_dir_mode}")

input_path = Path(INPUT_DIR)

# ------------------------------------------------------------
# Load canonical inputs
# ------------------------------------------------------------
# Canonical nodes table
nodes_path = input_path / "nodes_with_metrics.csv"
if nodes_path.exists():
    nodes_df = pd.read_csv(nodes_path)
    print(f"Loaded nodes_with_metrics.csv: {len(nodes_df)} nodes")
else:
    raise FileNotFoundError(f"Required file not found: {nodes_path}")

# Canonical cluster representatives
clusters_path = input_path / "top_papers_by_community.csv"
if clusters_path.exists():
    clusters_df = pd.read_csv(clusters_path)
    print(f"Loaded top_papers_by_community.csv: {len(clusters_df)} entries")
else:
    raise FileNotFoundError(f"Required file not found: {clusters_path}")

# ------------------------------------------------------------
# Post-load: Ensure openalex_wid exists (after nodes/clusters are loaded)
# ------------------------------------------------------------
if "openalex_wid" not in nodes_df.columns:
    if "openalex_id" in nodes_df.columns:
        nodes_df["openalex_wid"] = nodes_df["openalex_id"].apply(to_openalex_wid)
    elif "paper_id" in nodes_df.columns:
        nodes_df["openalex_wid"] = nodes_df["paper_id"].apply(to_openalex_wid)

if "openalex_wid" not in clusters_df.columns:
    if "openalex_id" in clusters_df.columns:
        clusters_df["openalex_wid"] = clusters_df["openalex_id"].apply(to_openalex_wid)
    elif "paper_id" in clusters_df.columns:
        clusters_df["openalex_wid"] = clusters_df["paper_id"].apply(to_openalex_wid)

# ------------------------------------------------------------
# Optional: load nodes_with_community for full cluster assignments
#   (requires input_path, so place after INPUT_DIR is resolved)
# ------------------------------------------------------------
assign_df = None
nodes_with_comm_path = input_path / "nodes_with_community.csv"
if nodes_with_comm_path.exists():
    assign_df = pd.read_csv(nodes_with_comm_path)

    # NOTE: standardize_id_column is defined later in your cell.
    # So here we do a minimal ID normalization without calling it.
    if "paper_id" not in assign_df.columns:
        for col in ["openalex_id", "openalex_wid", "id"]:
            if col in assign_df.columns:
                assign_df["paper_id"] = assign_df[col]
                break

    if "openalex_wid" not in assign_df.columns:
        if "openalex_id" in assign_df.columns:
            assign_df["openalex_wid"] = assign_df["openalex_id"].apply(to_openalex_wid)
        elif "paper_id" in assign_df.columns:
            assign_df["openalex_wid"] = assign_df["paper_id"].apply(to_openalex_wid)

    print(f"Loaded nodes_with_community.csv: {len(assign_df)} assignments")
else:
    print("nodes_with_community.csv not found (optional)")

# normalize cluster column name (robust)
if assign_df is not None:
    if "cluster" not in assign_df.columns:
        # common variants seen in exports
        candidate_cols = [
            "community", "cluster_id", "cluster",
            "community_id", "community_label",
            "group", "group_id",
            "partition", "partition_id",
            "louvain", "louvain_id",
            "leiden", "leiden_id",
            "modularity_class",
        ]
        for c in candidate_cols:
            if c in assign_df.columns:
                assign_df["cluster"] = assign_df[c]
                print(f"  [assign_df] Using '{c}' as cluster")
                break

    if "cluster" not in assign_df.columns:
        # Don't hard-fail: show columns to debug and proceed without assignments
        warnings.warn(
            f"nodes_with_community.csv has no recognized cluster column. "
            f"Available columns: {list(assign_df.columns)}. "
            f"Proceeding with assign_df but without 'cluster'.",
            UserWarning
        )

        
# Optional: top papers overall
top_overall_path = input_path / "top_papers_overall.csv"
if top_overall_path.exists():
    top_overall_df = pd.read_csv(top_overall_path)
    print(f"Loaded top_papers_overall.csv: {len(top_overall_df)} papers")
else:
    top_overall_df = None
    print("top_papers_overall.csv not found (optional, continuing)")

# Optional: edges
edges_df = None
for edge_file in ["edges.csv", "edges_viz.csv"]:
    edge_path = input_path / edge_file
    if edge_path.exists():
        edges_df = pd.read_csv(edge_path)
        print(f"Loaded {edge_file}: {len(edges_df)} edges")
        break

if edges_df is None:
    warnings.warn("No edges.csv or edges_viz.csv found (optional, continuing without edges)")

# ------------------------------------------------------------
# Optional: structured summaries (file-based) OR fallback from abstract
# ------------------------------------------------------------
summaries_df = None

# 1) Try to load structured summaries if present
for summary_file in ["summaries.csv", "paper_summaries.csv", "summaries.json"]:
    summary_path = input_path / summary_file
    if summary_path.exists():
        if summary_file.endswith(".csv"):
            summaries_df = pd.read_csv(summary_path)
        else:
            with open(summary_path, "r", encoding="utf-8") as f:
                summaries_data = json.load(f)

            # If it's a dict keyed by id -> summary object, normalize to rows
            if isinstance(summaries_data, dict):
                rows = []
                for k, v in summaries_data.items():
                    if isinstance(v, dict):
                        row = {"paper_id": k}
                        row.update(v)
                        rows.append(row)
                    else:
                        rows.append({"paper_id": k, "summary": v})
                summaries_df = pd.DataFrame(rows)
            else:
                summaries_df = pd.DataFrame(summaries_data)

        print(f"Loaded {summary_file}: {len(summaries_df)} summaries")
        break

# 2) Fallback: build summaries_df from nodes_df.abstract
if summaries_df is None:
    if "abstract" in nodes_df.columns and nodes_df["abstract"].notna().any():
        tmp = nodes_df[["openalex_wid", "abstract"]].copy()
        tmp = tmp.rename(columns={"openalex_wid": "paper_id", "abstract": "summary"})
        tmp = tmp.dropna(subset=["summary"]).drop_duplicates("paper_id")
        summaries_df = tmp.reset_index(drop=True)
        print(f"[Fallback] Built summaries_df from nodes_df.abstract: {len(summaries_df)} rows")
    else:
        summaries_df = pd.DataFrame(columns=["paper_id", "summary"])
        print("[Fallback] No structured summaries and no abstracts available. summaries_df is empty.")


# ------------------------------------------------------------
# Standardize identifier column name
# ------------------------------------------------------------
# Ensure we have a stable identifier across all dataframes
# Priority: openalex_id > openalex_wid > paper_id > id

def standardize_id_column(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Ensure df has a 'paper_id' column."""
    df = df.copy()
    
    for col in ['openalex_wid', 'openalex_id', 'paper_id', 'id']:
        if col in df.columns:
            df['paper_id'] = df[col]
            print(f"  [{name}] Using '{col}' as paper_id")
            return df
    
    raise ValueError(f"No identifier column found in {name}")

nodes_df = standardize_id_column(nodes_df, "nodes_df")
clusters_df = standardize_id_column(clusters_df, "clusters_df")

if summaries_df is not None:
    summaries_df = standardize_id_column(summaries_df, "summaries_df")

# ------------------------------------------------------------
# Optional: Day20 enrichment (abstract / concepts_top / rq_score ...)
# ------------------------------------------------------------
def find_latest_day20_run(root: str) -> Optional[Path]:
    p = Path(root)
    if not p.exists():
        return None
    run_dirs = [d for d in p.iterdir() if d.is_dir() and re.fullmatch(r"\d{8}_\d{6}", d.name)]
    if not run_dirs:
        return None
    run_dirs.sort(key=lambda d: d.name)
    return run_dirs[-1]

day20_df = None
DAY20_RUN_DIR = None

if getattr(CFG, "enable_day20_enrichment", False):
    DAY20_RUN_DIR = find_latest_day20_run(CFG.day20_artifacts_root)
    if DAY20_RUN_DIR is None:
        warnings.warn(f"Day20 enrichment enabled but no run dirs found under {CFG.day20_artifacts_root}")
    else:
        cand_path = DAY20_RUN_DIR / CFG.day20_enriched_pool_filename
        if not cand_path.exists():
            warnings.warn(f"Day20 enrichment enabled but file not found: {cand_path}")
        else:
            day20_df = pd.read_csv(cand_path)
            print(f"[Day20] Loaded enrichment pool: {cand_path} ({len(day20_df)} rows)")

# Build join key(s)
if day20_df is not None:
    # Ensure openalex_wid in day20_df
    if "openalex_wid" not in day20_df.columns:
        if "openalex_id" in day20_df.columns:
            day20_df["openalex_wid"] = day20_df["openalex_id"].apply(to_openalex_wid)

    # Keep only useful enrichment cols (存在するものだけ)
    want_cols = ["openalex_wid", "abstract", "concepts_top", "rq_score", "title", "doi"]
    keep_cols = [c for c in want_cols if c in day20_df.columns]
    day20_keep = day20_df[keep_cols].copy()

    # De-dupe Day20 by wid (keep richest)
    def _rich(row):
        score = 0
        for c in ["abstract", "concepts_top", "rq_score", "doi", "title"]:
            if c in row.index and pd.notna(row[c]):
                score += 1
        return score
    if "openalex_wid" in day20_keep.columns:
        day20_keep["_rich"] = day20_keep.apply(_rich, axis=1)
        day20_keep = day20_keep.sort_values("_rich", ascending=False).drop_duplicates("openalex_wid").drop(columns=["_rich"])

    # Enrich nodes_df by openalex_wid
    if "openalex_wid" in nodes_df.columns and "openalex_wid" in day20_keep.columns:
        before_nonnull_abs = nodes_df["abstract"].notna().sum() if "abstract" in nodes_df.columns else 0

        nodes_df = nodes_df.merge(
            day20_keep,
            on="openalex_wid",
            how="left",
            suffixes=("", "_day20"),
        )

        # If nodes_df already had abstract, prefer existing; else use day20
        if "abstract" in nodes_df.columns and "abstract_day20" in nodes_df.columns:
            nodes_df["abstract"] = nodes_df["abstract"].fillna(nodes_df["abstract_day20"])
            nodes_df.drop(columns=["abstract_day20"], inplace=True)

        if "concepts_top" in nodes_df.columns and "concepts_top_day20" in nodes_df.columns:
            nodes_df["concepts_top"] = nodes_df["concepts_top"].fillna(nodes_df["concepts_top_day20"])
            nodes_df.drop(columns=["concepts_top_day20"], inplace=True)

        if "rq_score" in nodes_df.columns and "rq_score_day20" in nodes_df.columns:
            nodes_df["rq_score"] = nodes_df["rq_score"].fillna(nodes_df["rq_score_day20"])
            nodes_df.drop(columns=["rq_score_day20"], inplace=True)

        after_nonnull_abs = nodes_df["abstract"].notna().sum() if "abstract" in nodes_df.columns else 0
        print(f"[Day20] Enriched nodes_df.abstract non-null: {before_nonnull_abs} -> {after_nonnull_abs}")
    else:
        warnings.warn("[Day20] openalex_wid missing in nodes_df or day20_df; enrichment skipped")

# ------------------------------------------------------------
# Fallback summaries AFTER Day20 enrichment (important)
# ------------------------------------------------------------
if (summaries_df is None or summaries_df.empty) and ("abstract" in nodes_df.columns) and nodes_df["abstract"].notna().any():
    tmp = nodes_df[["openalex_wid", "abstract"]].copy()
    tmp = tmp.rename(columns={"openalex_wid": "paper_id", "abstract": "summary"})
    tmp = tmp.dropna(subset=["summary"]).drop_duplicates("paper_id")
    summaries_df = tmp.reset_index(drop=True)
    print(f"[Fallback-after-Day20] Built summaries_df from nodes_df.abstract: {len(summaries_df)} rows")

print(f"\nInput loading complete")
print(f"- Nodes: {len(nodes_df)}")
print(f"- Cluster representatives: {len(clusters_df)}")
print(f"- Edges: {len(edges_df) if edges_df is not None else 'N/A'}")
print(f"- Summaries: {len(summaries_df) if summaries_df is not None else 'N/A'}")

Resolved INPUT_DIR (latest): artifacts/day21_paper_graph_20260114_094608
Loaded nodes_with_metrics.csv: 932 nodes
Loaded top_papers_by_community.csv: 151 entries
Loaded nodes_with_community.csv: 932 assignments
  [assign_df] Using 'community_id' as cluster
Loaded top_papers_overall.csv: 50 papers
Loaded edges.csv: 2773 edges
[Fallback] No structured summaries and no abstracts available. summaries_df is empty.
  [nodes_df] Using 'openalex_wid' as paper_id
  [clusters_df] Using 'openalex_wid' as paper_id
  [summaries_df] Using 'paper_id' as paper_id
[Day20] Loaded enrichment pool: artifacts/day20/20260113_061442/candidate_pool_enriched_scored.csv (902 rows)
[Day20] Enriched nodes_df.abstract non-null: 0 -> 541
[Fallback-after-Day20] Built summaries_df from nodes_df.abstract: 541 rows

Input loading complete
- Nodes: 932
- Cluster representatives: 151
- Edges: 2773
- Summaries: 541


In [5]:
# ============================================================
# Cell 03 — Select Representative Papers per Cluster
# ============================================================
# Overview:
# - Use top_papers_by_community.csv (clusters_df) as the primary source of representatives
# - Optionally re-rank within each cluster using available metrics
# - Ensure diversity: central + highly cited + recent + classic
# - Select K representative papers per cluster
#
# Inputs / Outputs:
# - Inputs: clusters_df (top_papers_by_community), nodes_df (nodes_with_metrics), assign_df (optional)
# - Outputs: representatives_df (community_id, paper_id, rank, diversity_tag, key metadata)
#
# Notes:
# - K set via CFG.k_representatives
# - clusters_df already contains per-community top candidates; this cell refines / diversifies

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 0) Validate required columns
# ------------------------------------------------------------
required = ["community_id", "openalex_wid"]
missing = [c for c in required if c not in clusters_df.columns]
if missing:
    raise ValueError(f"clusters_df missing required columns: {missing}")

# Use openalex_wid as stable paper_id
clusters_df = clusters_df.copy()
clusters_df["paper_id"] = clusters_df["openalex_wid"]

# ------------------------------------------------------------
# 1) Enrich clusters_df with any missing fields from nodes_df
#    (abstract / concepts_top / additional metrics, if present)
# ------------------------------------------------------------
nodes_key_cols = ["openalex_wid"]
# only merge columns that exist and are helpful
want_from_nodes = [
    "openalex_wid",
    "abstract",
    "concepts_top",
    "cited_by_count",
    "pagerank",
    "in_degree",
    "out_degree",
    "total_degree",
    "publication_year",
    "venue",
    "title",
    "rq_score",
    "priority_score",
]
nodes_merge_cols = [c for c in want_from_nodes if c in nodes_df.columns]
nodes_merge = nodes_df[nodes_merge_cols].drop_duplicates("openalex_wid").copy()

cluster_papers = clusters_df.merge(
    nodes_merge,
    on="openalex_wid",
    how="left",
    suffixes=("", "_node"),
)

print(f"cluster_papers rows: {len(cluster_papers)} (from clusters_df={len(clusters_df)})")

# ------------------------------------------------------------
# 2) Identify ranking columns (use best available)
# ------------------------------------------------------------
# Prefer pagerank if present (it exists in clusters_df per your columns)
rank_primary = None
for c in ["pagerank", "pagerank_node", "total_degree", "in_degree", "cited_by_count", "priority_score", "rq_score"]:
    if c in cluster_papers.columns:
        rank_primary = c
        break

if rank_primary is None:
    # fall back to no ranking
    print("[WARN] No ranking metric found; will keep input order.")
else:
    print(f"Primary ranking metric: {rank_primary}")

# choose year column
year_col = "publication_year" if "publication_year" in cluster_papers.columns else None

# ------------------------------------------------------------
# 3) Select K representatives per community_id with diversity tags
# ------------------------------------------------------------
def _pick_unique(df, n, sort_col=None, ascending=False, already=None):
    already = already or set()
    if sort_col and sort_col in df.columns:
        df2 = df.sort_values(sort_col, ascending=ascending)
    else:
        df2 = df
    picks = []
    for _, r in df2.iterrows():
        pid = r["paper_id"]
        if pid in already:
            continue
        picks.append(pid)
        already.add(pid)
        if len(picks) >= n:
            break
    return picks, already

def select_representatives(group_df: pd.DataFrame, k: int) -> pd.DataFrame:
    g = group_df.copy()

    # If fewer than k, label all
    if len(g) <= k:
        g = g.copy()
        g = g.sort_values(rank_primary, ascending=False) if rank_primary else g
        g["rank"] = range(1, len(g) + 1)
        g["diversity_tag"] = "all_selected"
        return g

    chosen = []
    chosen_set = set()

    # 1) central (top pagerank / degree)
    if rank_primary:
        n1 = max(1, k // 3)
        picks, chosen_set = _pick_unique(g, n1, sort_col=rank_primary, ascending=False, already=chosen_set)
        chosen += [(pid, "central") for pid in picks]

    # 2) highly cited (use cited_by_count if present)
    if "cited_by_count" in g.columns:
        n2 = max(1, k // 3)
        picks, chosen_set = _pick_unique(g, n2, sort_col="cited_by_count", ascending=False, already=chosen_set)
        chosen += [(pid, "highly_cited") for pid in picks]

    # 3) temporal diversity (recent + classic)
    if year_col and year_col in g.columns:
        n_recent = max(1, k // 6)
        n_classic = max(1, k // 6)

        picks, chosen_set = _pick_unique(g, n_recent, sort_col=year_col, ascending=False, already=chosen_set)
        chosen += [(pid, "recent") for pid in picks]

        picks, chosen_set = _pick_unique(g, n_classic, sort_col=year_col, ascending=True, already=chosen_set)
        chosen += [(pid, "classic") for pid in picks]

    # 4) fill remaining with top-ranked
    remain = k - len(chosen)
    if remain > 0:
        picks, chosen_set = _pick_unique(g, remain, sort_col=rank_primary, ascending=False, already=chosen_set)
        chosen += [(pid, "top_ranked") for pid in picks]

    # Build output rows in chosen order
    rows = []
    for rank, (pid, tag) in enumerate(chosen[:k], 1):
        row = g[g["paper_id"] == pid].iloc[0].to_dict()
        row["rank"] = rank
        row["diversity_tag"] = tag
        rows.append(row)

    return pd.DataFrame(rows)

representatives = []
for cid, grp in cluster_papers.groupby("community_id"):
    reps = select_representatives(grp, CFG.k_representatives)
    reps["community_id"] = cid
    representatives.append(reps)

representatives_df = pd.concat(representatives, ignore_index=True)

# ------------------------------------------------------------
# 4) Light cleanup: keep a stable minimal schema (optional)
# ------------------------------------------------------------
# Keep useful columns if they exist
keep_order = [
    "community_id",
    "paper_id",
    "openalex_wid",
    "openalex_id",
    "title",
    "publication_year",
    "venue",
    "pagerank",
    "cited_by_count",
    "rq_score",
    "priority_score",
    "diversity_tag",
    "rank",
]
final_cols = [c for c in keep_order if c in representatives_df.columns] + \
             [c for c in representatives_df.columns if c not in keep_order]
representatives_df = representatives_df[final_cols].copy()

print(
    f"Selected {len(representatives_df)} representatives across "
    f"{representatives_df['community_id'].nunique()} communities "
    f"(K={CFG.k_representatives})."
)

# Save
repr_path = Path(CFG.output_dir) / "representatives_per_cluster.csv"
representatives_df.to_csv(repr_path, index=False)
print(f"Saved: {repr_path}")

display(representatives_df.head(10))


cluster_papers rows: 151 (from clusters_df=151)
Primary ranking metric: pagerank
Selected 80 representatives across 8 communities (K=10).
Saved: artifacts/day22_gap_mining/20260118_142302/representatives_per_cluster.csv


,community_id,paper_id,openalex_wid,openalex_id,title,publication_year,venue,pagerank,cited_by_count,rq_score,...,concepts_top,pagerank_node,in_degree_node,out_degree_node,total_degree_node,publication_year_node,venue_node,title_node,rq_score_node,priority_score_node
0,0,W1983987175,W1983987175,https://openalex.org/W1983987175,Syndicated investments by venture capital firm...,1987.0,Journal of Business Venturing,0.016489,399,0.019231,...,Venture capital; Social venture capital; Busin...,0.016489,20,1,21,1987.0,Journal of Business Venturing,Syndicated investments by venture capital firm...,0.019231,0.331139
1,0,W2078266734,W2078266734,https://openalex.org/W2078266734,The structure of the investment networks of ve...,1988.0,Journal of Business Venturing,0.014869,248,0.019231,...,Venture capital; Competitor analysis; Portfoli...,0.014869,8,1,9,1988.0,Journal of Business Venturing,The structure of the investment networks of ve...,0.019231,0.249943
2,0,W1988247859,W1988247859,https://openalex.org/W1988247859,Route 128: The development of a regional high ...,1983.0,Research Policy,0.007356,368,0.007692,...,Economies of agglomeration; State (computer sc...,0.007356,3,0,3,1983.0,Research Policy,Route 128: The development of a regional high ...,0.007692,0.260827
3,0,W4297081765,W4297081765,https://openalex.org/W4297081765,Harvard business review,1994.0,Journal of the American Dietetic Association,0.005505,46507,0.000000,...,Medicine,0.005505,19,0,19,1994.0,Journal of the American Dietetic Association,Harvard business review,0.000000,0.383513
4,0,W2162484441,W2162484441,https://openalex.org/W2162484441,Endogenous Technological Change,1990.0,Journal of Political Economy,0.003863,15199,0.023077,...,Monopolistic competition; Economics; Excludabi...,0.003863,7,0,7,1990.0,Journal of Political Economy,Endogenous Technological Change,0.023077,0.387104
5,0,W2130136629,W2130136629,https://openalex.org/W2130136629,Journal of economic literature,1996.0,International Journal of Forecasting,0.002653,9980,0.000000,...,Economics; Regional science; Sociology,0.002653,7,0,7,1996.0,International Journal of Forecasting,Journal of economic literature,0.000000,0.327510
6,0,W2017370934,W2017370934,https://openalex.org/W2017370934,Creating good public policy to support high-gr...,2011.0,Small Business Economics,0.001930,552,0.011538,...,Argument (complex analysis); Entrepreneurship;...,0.001930,9,4,13,2011.0,Small Business Economics,Creating good public policy to support high-gr...,0.011538,0.217052
7,0,W2087194317,W2087194317,https://openalex.org/W2087194317,Power and Centrality: A Family of Measures,1987.0,American Journal of Sociology,0.003460,5041,0.003846,...,Centrality; Generalization; Power (physics); N...,0.003460,13,0,13,1987.0,American Journal of Sociology,Power and Centrality: A Family of Measures,0.003846,0.370716
8,0,W2165429918,W2165429918,https://openalex.org/W2165429918,Geographic Localization of Knowledge Spillover...,1993.0,The Quarterly Journal of Economics,0.007353,7604,0.000000,...,Economic geography; State (computer science); ...,0.007353,21,1,22,1993.0,The Quarterly Journal of Economics,Geographic Localization of Knowledge Spillover...,0.000000,0.325068
9,0,W2082800434,W2082800434,https://openalex.org/W2082800434,The Syndication of Venture Capital Investments,1994.0,Financial Management,0.005811,839,0.019231,...,Web syndication; Venture capital; Business; So...,0.005811,36,3,39,1994.0,Financial Management,The Syndication of Venture Capital Investments,0.019231,0.340441


,community_id,paper_id,openalex_wid,openalex_id,title,publication_year,venue,pagerank,cited_by_count,rq_score,...,concepts_top,pagerank_node,in_degree_node,out_degree_node,total_degree_node,publication_year_node,venue_node,title_node,rq_score_node,priority_score_node
0,0,W1983987175,W1983987175,https://openalex.org/W1983987175,Syndicated investments by venture capital firm...,1987.0,Journal of Business Venturing,0.016489,399,0.019231,...,Venture capital; Social venture capital; Busin...,0.016489,20,1,21,1987.0,Journal of Business Venturing,Syndicated investments by venture capital firm...,0.019231,0.331139
1,0,W2078266734,W2078266734,https://openalex.org/W2078266734,The structure of the investment networks of ve...,1988.0,Journal of Business Venturing,0.014869,248,0.019231,...,Venture capital; Competitor analysis; Portfoli...,0.014869,8,1,9,1988.0,Journal of Business Venturing,The structure of the investment networks of ve...,0.019231,0.249943
2,0,W1988247859,W1988247859,https://openalex.org/W1988247859,Route 128: The development of a regional high ...,1983.0,Research Policy,0.007356,368,0.007692,...,Economies of agglomeration; State (computer sc...,0.007356,3,0,3,1983.0,Research Policy,Route 128: The development of a regional high ...,0.007692,0.260827
3,0,W4297081765,W4297081765,https://openalex.org/W4297081765,Harvard business review,1994.0,Journal of the American Dietetic Association,0.005505,46507,0.000000,...,Medicine,0.005505,19,0,19,1994.0,Journal of the American Dietetic Association,Harvard business review,0.000000,0.383513
4,0,W2162484441,W2162484441,https://openalex.org/W2162484441,Endogenous Technological Change,1990.0,Journal of Political Economy,0.003863,15199,0.023077,...,Monopolistic competition; Economics; Excludabi...,0.003863,7,0,7,1990.0,Journal of Political Economy,Endogenous Technological Change,0.023077,0.387104
5,0,W2130136629,W2130136629,https://openalex.org/W2130136629,Journal of economic literature,1996.0,International Journal of Forecasting,0.002653,9980,0.000000,...,Economics; Regional science; Sociology,0.002653,7,0,7,1996.0,International Journal of Forecasting,Journal of economic literature,0.000000,0.327510
6,0,W2017370934,W2017370934,https://openalex.org/W2017370934,Creating good public policy to support high-gr...,2011.0,Small Business Economics,0.001930,552,0.011538,...,Argument (complex analysis); Entrepreneurship;...,0.001930,9,4,13,2011.0,Small Business Economics,Creating good public policy to support high-gr...,0.011538,0.217052
7,0,W2087194317,W2087194317,https://openalex.org/W2087194317,Power and Centrality: A Family of Measures,1987.0,American Journal of Sociology,0.003460,5041,0.003846,...,Centrality; Generalization; Power (physics); N...,0.003460,13,0,13,1987.0,American Journal of Sociology,Power and Centrality: A Family of Measures,0.003846,0.370716
8,0,W2165429918,W2165429918,https://openalex.org/W2165429918,Geographic Localization of Knowledge Spillover...,1993.0,The Quarterly Journal of Economics,0.007353,7604,0.000000,...,Economic geography; State (computer science); ...,0.007353,21,1,22,1993.0,The Quarterly Journal of Economics,Geographic Localization of Knowledge Spillover...,0.000000,0.325068
9,0,W2082800434,W2082800434,https://openalex.org/W2082800434,The Syndication of Venture Capital Investments,1994.0,Financial Management,0.005811,839,0.019231,...,Web syndication; Venture capital; Business; So...,0.005811,36,3,39,1994.0,Financial Management,The Syndication of Venture Capital Investments,0.019231,0.340441


In [6]:
# ============================================================
# Cell 04 — Build Evidence Packs for Gap Extraction
# ============================================================
# Overview:
# - For each representative paper, build an evidence pack
# - Include: (fallback) summary text, bibliographic info, centrality/ranking signals
# - Control length for token efficiency
#
# Inputs / Outputs:
# - Inputs: representatives_df, summaries_df (paper_id -> summary), nodes_df (optional, for abstract/concepts_top)
# - Outputs: evidence_packs (dict: community_id -> list of evidence dicts)
#
# Notes:
# - representatives_df is grouped by community_id (NOT cluster_id)
# - summaries_df is expected to have columns: paper_id, summary
# - If structured summary fields exist, include them; else use summary/abstract

import json
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Prep: fast lookup for summaries (avoid per-row DataFrame filter)
# ------------------------------------------------------------
summary_lookup = {}
if summaries_df is not None and not summaries_df.empty:
    # Prefer 'summary' column; if not present, try 'abstract'
    text_col = "summary" if "summary" in summaries_df.columns else ("abstract" if "abstract" in summaries_df.columns else None)
    if text_col is not None:
        summary_lookup = (
            summaries_df.dropna(subset=["paper_id", text_col])
            .drop_duplicates("paper_id")
            .set_index("paper_id")[text_col]
            .to_dict()
        )
    else:
        summary_lookup = {}

# Optional: concepts lookup from nodes_df if present
concepts_lookup = {}
if "concepts_top" in nodes_df.columns:
    tmp = nodes_df[["openalex_wid", "concepts_top"]].dropna().drop_duplicates("openalex_wid")
    concepts_lookup = tmp.set_index("openalex_wid")["concepts_top"].to_dict()

# ------------------------------------------------------------
# Helpers: safe conversion + truncation
# ------------------------------------------------------------
def _s(x):
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    s = str(x).strip()
    return s if s else None

def _truncate(s: str, n: int) -> str:
    s = s or ""
    s = s.strip()
    return s[:n]

# ------------------------------------------------------------
# Build evidence for each paper
# ------------------------------------------------------------
def build_paper_evidence(row: pd.Series) -> dict:
    """Build evidence pack for a single paper (row from representatives_df)."""
    paper_id = row.get("paper_id", None)  # openalex_wid
    wid = row.get("openalex_wid", paper_id)

    evidence = {
        "paper_id": _s(paper_id),
        "rank": int(row["rank"]) if "rank" in row and pd.notna(row["rank"]) else None,
        "diversity_tag": _s(row.get("diversity_tag", None)),
    }

    # Bibliographic info
    title = _s(row.get("title", None))
    if title:
        evidence["title"] = _truncate(title, 200)

    year = row.get("publication_year", None)
    if pd.notna(year):
        try:
            evidence["year"] = int(year)
        except Exception:
            pass

    venue = _s(row.get("venue", None))
    if venue:
        evidence["venue"] = _truncate(venue, 120)

    # Citations (your data uses cited_by_count)
    cbc = row.get("cited_by_count", None)
    if pd.notna(cbc):
        try:
            evidence["citations"] = int(cbc)
        except Exception:
            pass

    # Centrality / graph metrics (use what's actually present)
    for metric in ["pagerank", "in_degree", "out_degree", "total_degree"]:
        if metric in row and pd.notna(row[metric]):
            try:
                evidence[metric] = float(row[metric])
            except Exception:
                pass

    # Summary text policy:
    # - Prefer structured summaries if present (contribution/method/... columns)
    # - Else: use summaries_df['summary'] (which is abstract fallback)
    # - Else: use row['abstract'] if present
    # We store it under "summary_text" to avoid confusion.
    summary_text = None

    # A) structured fields if they exist in row (rare, but keep)
    structured_fields = ["contribution", "method", "results", "limitations", "future_work"]
    has_structured = any((f in row.index and pd.notna(row[f])) for f in structured_fields)
    if has_structured:
        for f in structured_fields:
            if f in row.index and pd.notna(row[f]):
                evidence[f] = _truncate(_s(row[f]) or "", 300)

    # B) summary_lookup (paper_id is openalex_wid)
    if not summary_text and paper_id in summary_lookup:
        summary_text = _s(summary_lookup.get(paper_id))

    # C) row abstract
    if not summary_text:
        summary_text = _s(row.get("abstract", None))

    if summary_text:
        evidence["summary_text"] = _truncate(summary_text, 800)

    # Concepts (optional enrichment)
    if wid in concepts_lookup:
        evidence["concepts_top"] = concepts_lookup[wid]

    return evidence

# ------------------------------------------------------------
# Group evidence packs by community (cluster)
# ------------------------------------------------------------
if "community_id" not in representatives_df.columns:
    raise ValueError("representatives_df must have 'community_id' column (from Cell 03)")

evidence_packs: dict = {}

for community_id, group in representatives_df.groupby("community_id"):
    cluster_evidence = []
    for _, row in group.iterrows():
        cluster_evidence.append(build_paper_evidence(row))
    evidence_packs[str(community_id)] = cluster_evidence

print(f"\nBuilt evidence packs for {len(evidence_packs)} communities")
print(f"Total evidence entries: {sum(len(v) for v in evidence_packs.values())}")

# Quick sanity check: how many have summary_text?
n_with_text = sum(1 for v in evidence_packs.values() for e in v if _s(e.get("summary_text")) is not None)
print(f"Evidence entries with summary_text: {n_with_text}")

# Save
evidence_path = Path(CFG.output_dir) / "evidence_packs.json"
with open(evidence_path, "w", encoding="utf-8") as f:
    json.dump(evidence_packs, f, ensure_ascii=False, indent=2)

print(f"Saved: {evidence_path}")



Built evidence packs for 8 communities
Total evidence entries: 80
Evidence entries with summary_text: 48
Saved: artifacts/day22_gap_mining/20260118_142302/evidence_packs.json


In [10]:
# ============================================================
# Cell 05 — (Private Hook) Gap Extraction Prompt Template (patched v2)
# ============================================================

from pathlib import Path
import warnings

PUBLIC_PLACEHOLDER_GAP_PROMPT = """You are a research analyst identifying unresolved research gaps from a cluster of papers.

Given a cluster context and evidence from representative papers, identify 3-5 major unresolved research gaps.

For each gap, provide:
1. gap_id: unique identifier (e.g., "gap_cluster_X_001")
2. gap_statement: clear description of the unresolved issue
3. gap_type: one of "coverage" (uncovered areas), "measurement" (data/metric difficulties), or "causal" (causal inference challenges)
4. evidence_papers: list of paper IDs that support this gap
5. falsifiable_test: a concrete, testable hypothesis that would address this gap
6. counterarguments: potential reasons why this gap might not be critical

Return your answer as valid JSON with structure:
{
  "gaps": [
    {
      "gap_id": "...",
      "gap_statement": "...",
      "gap_type": "...",
      "evidence_papers": [...],
      "falsifiable_test": "...",
      "counterarguments": "..."
    }
  ]
}

Cluster context:
{cluster_context}

Representative papers evidence:
{papers_evidence}
"""

# ---- robust path resolving ----
project_root = Path(getattr(CFG, "project_root", Path.cwd())).resolve()
raw_path = Path(CFG.prompt_gap_extract_path)
prompt_path = raw_path if raw_path.is_absolute() else (project_root / raw_path).resolve()

PROMPT_GAP_EXTRACT = None

if prompt_path.exists():
    PROMPT_GAP_EXTRACT = prompt_path.read_text(encoding="utf-8")
    print(f"Loaded gap extraction prompt: {prompt_path}")
else:
    created = False

    # Try to create a template file for the user to edit
    try:
        prompt_path.parent.mkdir(parents=True, exist_ok=True)
        prompt_path.write_text(PUBLIC_PLACEHOLDER_GAP_PROMPT, encoding="utf-8")
        created = True
        print(f"Created placeholder prompt file (edit this for private version): {prompt_path}")
    except Exception as e:
        warnings.warn(f"Could not create prompt template file at {prompt_path}: {e}", stacklevel=2)

    # If creation succeeded (or file appeared), load it immediately
    if prompt_path.exists():
        PROMPT_GAP_EXTRACT = prompt_path.read_text(encoding="utf-8")
        print(f"Loaded gap extraction prompt: {prompt_path}")
    else:
        PROMPT_GAP_EXTRACT = PUBLIC_PLACEHOLDER_GAP_PROMPT
        warnings.warn(
            f"Prompt file not found. Using public placeholder. Looked for: {prompt_path}",
            stacklevel=2
        )

print(f"Gap extraction prompt ready ({len(PROMPT_GAP_EXTRACT)} chars)")


Loaded gap extraction prompt: /Users/yuetoya/Desktop/researchOS100-private/notebooks/prompts/private/gap_extraction.txt
Gap extraction prompt ready (1035 chars)


In [13]:
# ============================================================
# Cell 06 — Cluster-wise Gap Extraction (LLM)
# ============================================================
# Overview:
# - Run LLM per cluster using bundled evidence packs
# - Request structured JSON output with gap details
# - Retry until JSON is parseable
# - If dry_run=True, create placeholder outputs
#
# Inputs / Outputs:
# - Inputs: evidence_packs, PROMPT_GAP_EXTRACT
# - Outputs: extracted_gaps (list of gap dicts)
#
# Notes:
# - Uses OpenAI gpt-4o-mini with temperature 0.0
# - Validates JSON structure before accepting

# ------------------------------------------------------------
# Initialize OpenAI client
# ------------------------------------------------------------
client = None
if 'CFG' in locals() and hasattr(CFG, 'dry_run') and not CFG.dry_run:
    client = OpenAI()
    print("OpenAI client initialized")

# ------------------------------------------------------------
# Gap extraction function
# ------------------------------------------------------------
def extract_gaps_for_cluster(
    cluster_id: str,
    evidence_list: List[Dict],
    max_retries: int = 3
) -> List[Dict]:
    """Extract gaps for a single cluster using LLM."""
    
    if 'CFG' in locals() and hasattr(CFG, 'dry_run') and CFG.dry_run:
        # Create placeholder gaps
        return [
            {
                "gap_id": f"gap_cluster_{cluster_id}_001",
                "gap_statement": f"[DRY RUN] Placeholder gap for cluster {cluster_id}",
                "gap_type": "coverage",
                "evidence_papers": [ev.get('paper_id', 'unknown') for ev in evidence_list[:3]],
                "falsifiable_test": "[DRY RUN] Placeholder test",
                "counterarguments": "[DRY RUN] Placeholder counterarguments"
            }
        ]
    
    # Build cluster context
    cluster_context = f"Cluster ID: {cluster_id}\n"
    cluster_context += f"Number of representative papers: {len(evidence_list)}\n"
    
    # Build papers evidence
    papers_evidence = ""
    for i, ev in enumerate(evidence_list, 1):
        papers_evidence += f"\n--- Paper {i} ---\n"
        papers_evidence += f"ID: {ev.get('paper_id', 'unknown')}\n"
        if 'title' in ev:
            papers_evidence += f"Title: {ev['title']}\n"
        if 'year' in ev:
            papers_evidence += f"Year: {ev['year']}\n"
        if 'citations' in ev:
            papers_evidence += f"Citations: {ev['citations']}\n"
        if 'abstract' in ev:
            papers_evidence += f"Abstract: {ev['abstract']}\n"
        if 'limitations' in ev:
            papers_evidence += f"Limitations: {ev['limitations']}\n"
        if 'future_work' in ev:
            papers_evidence += f"Future work: {ev['future_work']}\n"
    
    # Format prompt (safe replacement: avoids .format() collisions with JSON braces)
    prompt = (
        PROMPT_GAP_EXTRACT
        .replace("{cluster_context}", cluster_context)
        .replace("{papers_evidence}", papers_evidence)
    )

    
    # Call LLM with retries
    for attempt in range(max_retries):
        try:
            if client is not None:
                response = client.chat.completions.create(
                    model=getattr(CFG, "llm_model", "gpt-4o-mini"),
                    messages=[
                        {"role": "system", "content": "You are a research analyst. Always respond with valid JSON."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=getattr(CFG, "llm_temperature", 0.0),
                    response_format={"type": "json_object"},
                )
                content = response.choices[0].message.content.strip()
                result = json.loads(content)
            else:
                raise ValueError("Client not initialized")
                
            content = response.choices[0].message.content.strip()
            
            # Try to parse JSON
            # Handle potential markdown code blocks
            # Handle potential markdown code blocks (```json ... ```)
            if content.startswith("```"):
                content = content.strip().lstrip("`")
                # Better: strip fenced block robustly
                lines = response.choices[0].message.content.strip().splitlines()
                if lines and lines[0].startswith("```"):
                    lines = lines[1:]
                if lines and lines[-1].startswith("```"):
                    lines = lines[:-1]
                content = "\n".join(lines).strip()
                if content.lower().startswith("json"):
                    content = content[4:].strip()

            
            result = json.loads(content)
            
            # Validate structure
            if 'gaps' not in result:
                raise ValueError("Response missing 'gaps' key")
            
            gaps = result['gaps']
            
            # Add cluster_id to each gap
            for gap in gaps:
                gap['cluster_id'] = cluster_id
            
            return gaps
            
        except (json.JSONDecodeError, ValueError) as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} for cluster {cluster_id}: {e}")
            else:
                print(f"  Failed to parse JSON for cluster {cluster_id} after {max_retries} attempts")
                # Return empty list on failure
                return []
    
    return []

# ------------------------------------------------------------
# Extract gaps for all clusters
# ------------------------------------------------------------
print(f"\nExtracting gaps for {len(evidence_packs)} clusters...")
print(f"Dry run: {CFG.dry_run if hasattr(CFG, 'dry_run') else False}")

extracted_gaps = []

for i, (cluster_id, evidence_list) in enumerate(evidence_packs.items(), 1):
    print(f"Processing cluster {cluster_id} ({i}/{len(evidence_packs)})...")
    
    gaps = extract_gaps_for_cluster(cluster_id, evidence_list)
    extracted_gaps.extend(gaps)
    
    print(f"  Extracted {len(gaps)} gaps")

print(f"\nTotal gaps extracted: {len(extracted_gaps)}")

# Save raw extracted gaps
raw_gaps_path = Path(CFG.output_dir if hasattr(CFG, 'output_dir') else '.') / "gaps_raw.json"
with open(raw_gaps_path, 'w') as f:
    json.dump(extracted_gaps, f, indent=2)

print(f"Saved: {raw_gaps_path}")

OpenAI client initialized

Extracting gaps for 8 clusters...
Dry run: False
Processing cluster 0 (1/8)...
  Extracted 4 gaps
Processing cluster 1 (2/8)...
  Extracted 4 gaps
Processing cluster 2 (3/8)...
  Extracted 4 gaps
Processing cluster 3 (4/8)...
  Extracted 4 gaps
Processing cluster 4 (5/8)...
  Extracted 4 gaps
Processing cluster 5 (6/8)...
  Extracted 4 gaps
Processing cluster 6 (7/8)...
  Extracted 4 gaps
Processing cluster 7 (8/8)...
  Extracted 4 gaps

Total gaps extracted: 32
Saved: artifacts/day22_gap_mining/20260118_142302/gaps_raw.json


In [14]:
# ============================================================
# Cell 07 — Gap Normalization
# ============================================================
# Overview:
# - Merge near-duplicate gaps using string similarity
# - Normalize granularity (split broad gaps, merge narrow ones)
# - Tag cross-cluster gaps explicitly
#
# Inputs / Outputs:
# - Inputs: extracted_gaps
# - Outputs: normalized_gaps
#
# Notes:
# - Uses SequenceMatcher for similarity detection
# - Threshold set via CFG.similarity_threshold

# ------------------------------------------------------------
# Helper: Calculate string similarity
# ------------------------------------------------------------
def similarity_score(str1: str, str2: str) -> float:
    """Calculate similarity between two strings (0-1)."""
    return SequenceMatcher(None, str1.lower(), str2.lower()).ratio()

# ------------------------------------------------------------
# Merge near-duplicate gaps
# ------------------------------------------------------------
def merge_duplicates(gaps: List[Dict], threshold: float) -> List[Dict]:
    """Merge gaps with similar gap_statement."""
    if len(gaps) == 0:
        return []
    
    # Track merged groups
    merged_groups = []
    used_indices = set()
    
    for i, gap1 in enumerate(gaps):
        if i in used_indices:
            continue
        
        # Start new group
        group = [gap1]
        used_indices.add(i)
        
        # Find similar gaps
        for j, gap2 in enumerate(gaps):
            if j <= i or j in used_indices:
                continue
            
            sim = similarity_score(gap1['gap_statement'], gap2['gap_statement'])
            
            if sim >= threshold:
                group.append(gap2)
                used_indices.add(j)
        
        merged_groups.append(group)
    
    # Merge each group
    normalized = []
    for group in merged_groups:
        if len(group) == 1:
            normalized.append(group[0])
        else:
            # Merge multiple gaps
            merged_gap = {
                'gap_id': group[0]['gap_id'],  # Use first gap's ID
                'gap_statement': group[0]['gap_statement'],  # Use first gap's statement
                'gap_type': group[0]['gap_type'],
                'evidence_papers': [],
                'falsifiable_test': group[0]['falsifiable_test'],
                'counterarguments': group[0]['counterarguments'],
                'cluster_id': [],
                'merged_from': []
            }
            
            # Aggregate evidence papers and clusters
            for gap in group:
                if 'evidence_papers' in gap:
                    merged_gap['evidence_papers'].extend(gap['evidence_papers'])
                
                if 'cluster_id' in gap:
                    if isinstance(gap['cluster_id'], list):
                        merged_gap['cluster_id'].extend(gap['cluster_id'])
                    else:
                        merged_gap['cluster_id'].append(gap['cluster_id'])
                
                merged_gap['merged_from'].append(gap['gap_id'])
            
            # Deduplicate
            merged_gap['evidence_papers'] = list(set(merged_gap['evidence_papers']))
            merged_gap['cluster_id'] = list(set(merged_gap['cluster_id']))
            
            normalized.append(merged_gap)
    
    return normalized

print(f"Merging near-duplicate gaps (threshold: {CFG.similarity_threshold})...")
normalized_gaps = merge_duplicates(extracted_gaps, CFG.similarity_threshold)

print(f"Gaps after normalization: {len(normalized_gaps)} (from {len(extracted_gaps)})")

# ------------------------------------------------------------
# Tag cross-cluster gaps
# ------------------------------------------------------------
for gap in normalized_gaps:
    # Ensure cluster_id is a list
    if 'cluster_id' in gap and not isinstance(gap['cluster_id'], list):
        gap['cluster_id'] = [gap['cluster_id']]
    
    # Tag cross-cluster gaps
    if 'cluster_id' in gap and len(gap['cluster_id']) > 1:
        gap['is_cross_cluster'] = True
    else:
        gap['is_cross_cluster'] = False

cross_cluster_count = sum(1 for g in normalized_gaps if g.get('is_cross_cluster', False))
print(f"Cross-cluster gaps: {cross_cluster_count}")

# Save normalized gaps
norm_gaps_path = Path(CFG.output_dir) / "gaps_normalized.json"
with open(norm_gaps_path, 'w') as f:
    json.dump(normalized_gaps, f, indent=2)

print(f"Saved: {norm_gaps_path}")

Merging near-duplicate gaps (threshold: 0.85)...
Gaps after normalization: 32 (from 32)
Cross-cluster gaps: 0
Saved: artifacts/day22_gap_mining/20260118_142302/gaps_normalized.json


In [15]:
# ============================================================
# Cell 08 — Gap Prioritization
# ============================================================
# Overview:
# - Score gaps on importance, feasibility, novelty
# - Importance: share of central papers in evidence
# - Feasibility: placeholder (updated after Cell 09)
# - Novelty: low overlap with past gaps (if available)
# - Rank gaps and output gaps_ranked.csv
#
# Inputs / Outputs:
# - Inputs: normalized_gaps, representatives_df, CFG.previous_gaps_path
# - Outputs: gaps_ranked_df
#
# Notes:
# - Weights configured via CFG.priority_weights
# - Feasibility placeholder until data mapping in Cell 09

# ------------------------------------------------------------
# Calculate importance score
# ------------------------------------------------------------
def calculate_importance(gap: Dict, representatives_df: pd.DataFrame) -> float:
    """Calculate importance based on centrality of evidence papers."""
    if 'evidence_papers' not in gap or len(gap['evidence_papers']) == 0:
        return 0.0
    
    # Get evidence papers
    evidence_ids = gap['evidence_papers']
    evidence_papers = representatives_df[representatives_df['paper_id'].isin(evidence_ids)]
    
    if len(evidence_papers) == 0:
        return 0.0
    
    # Use pagerank or degree_centrality if available
    centrality_col = None
    for col in ['pagerank', 'degree_centrality', 'betweenness_centrality']:
        if col in evidence_papers.columns:
            centrality_col = col
            break
    
    if centrality_col is None:
        # Fallback: use fraction of representative papers
        return len(evidence_papers) / len(representatives_df)
    
    # Average centrality of evidence papers
    avg_centrality = evidence_papers[centrality_col].mean()
    return float(avg_centrality)

# ------------------------------------------------------------
# Calculate novelty score
# ------------------------------------------------------------
def calculate_novelty(gap: Dict, previous_gaps: List[Dict]) -> float:
    """Calculate novelty vs previous gaps."""
    if previous_gaps is None or len(previous_gaps) == 0:
        return 1.0  # Max novelty if no previous gaps
    
    gap_statement = gap['gap_statement']
    
    # Find max similarity with any previous gap
    max_similarity = 0.0
    for prev_gap in previous_gaps:
        if 'gap_statement' in prev_gap:
            sim = similarity_score(gap_statement, prev_gap['gap_statement'])
            max_similarity = max(max_similarity, sim)
    
    # Novelty is inverse of similarity
    return 1.0 - max_similarity

# ------------------------------------------------------------
# Load previous gaps if available
# ------------------------------------------------------------
previous_gaps = None
if CFG.previous_gaps_path and Path(CFG.previous_gaps_path).exists():
    with open(CFG.previous_gaps_path, 'r') as f:
        previous_gaps = json.load(f)
    print(f"Loaded {len(previous_gaps)} previous gaps for novelty scoring")
else:
    print("No previous gaps available; novelty score will be 1.0 for all")

# ------------------------------------------------------------
# Score all gaps
# ------------------------------------------------------------
print(f"\nScoring {len(normalized_gaps)} gaps...")

for gap in normalized_gaps:
    # Importance
    gap['importance_score'] = calculate_importance(gap, representatives_df)
    
    # Feasibility (placeholder until Cell 09)
    gap['feasibility_score'] = 0.5  # Neutral default
    
    # Novelty
    gap['novelty_score'] = calculate_novelty(gap, previous_gaps)
    
    # Composite priority score
    gap['priority_score'] = (
        CFG.priority_weights['importance'] * gap['importance_score'] +
        CFG.priority_weights['feasibility'] * gap['feasibility_score'] +
        CFG.priority_weights['novelty'] * gap['novelty_score']
    )

# Sort by priority
normalized_gaps.sort(key=lambda x: x['priority_score'], reverse=True)

print("Scoring complete")
print(f"Top gap priority score: {normalized_gaps[0]['priority_score']:.3f}")
print(f"Median gap priority score: {normalized_gaps[len(normalized_gaps)//2]['priority_score']:.3f}")

# ------------------------------------------------------------
# Save ranked gaps
# ------------------------------------------------------------
# Convert to DataFrame
gaps_ranked_df = pd.DataFrame(normalized_gaps)

# Flatten list columns for CSV
for col in ['evidence_papers', 'cluster_id', 'merged_from']:
    if col in gaps_ranked_df.columns:
        gaps_ranked_df[col] = gaps_ranked_df[col].apply(
            lambda x: '|'.join(map(str, x)) if isinstance(x, list) else str(x)
        )

ranked_path = Path(CFG.output_dir) / "gaps_ranked.csv"
gaps_ranked_df.to_csv(ranked_path, index=False)
print(f"Saved: {ranked_path}")

# Also save JSON version
ranked_json_path = Path(CFG.output_dir) / "gaps_ranked.json"
with open(ranked_json_path, 'w') as f:
    json.dump(normalized_gaps, f, indent=2)
print(f"Saved: {ranked_json_path}")

No previous gaps available; novelty score will be 1.0 for all

Scoring 32 gaps...
Scoring complete
Top gap priority score: 0.359
Median gap priority score: 0.351
Saved: artifacts/day22_gap_mining/20260118_142302/gaps_ranked.csv
Saved: artifacts/day22_gap_mining/20260118_142302/gaps_ranked.json


In [17]:
# ============================================================
# Cell 09 — (Private Hook) Gap → Data Mapping
# ============================================================
# Overview:
# - Load gap-to-data mapping prompt from external file
# - For each gap, identify candidate data sources
# - Update feasibility score based on data availability
# - Allow "no data exists" as valid output
#
# Inputs / Outputs:
# - Inputs: normalized_gaps, PROMPT_DATA_MAP
# - Outputs: gaps with data_sources field, updated feasibility_score
#
# Notes:
# - Uses OpenAI gpt-4o-mini
# - If dry_run=True, create placeholder mappings

# ------------------------------------------------------------
# Load data mapping prompt (patched v2)
# ------------------------------------------------------------
from pathlib import Path
import warnings

PUBLIC_PLACEHOLDER_DATA_MAP_PROMPT = """You are a research data specialist. Given a research gap, identify potential data sources that could address it.

Gap details:
Type: {gap_type}
Statement: {gap_statement}
Falsifiable test: {falsifiable_test}

Identify candidate data sources from these categories:
- policy (government policies, regulations)
- investment (funding, VC, M&A)
- firm (company financials, operations)
- labor (employment, skills, wages)
- patent (patent filings, citations)
- geography (location-based data)
- text (documents, news, social media)

For each candidate source, provide:
1. source_category: one of the above
2. source_description: specific dataset or data type
3. availability: "public", "restricted", or "unavailable"
4. collection_difficulty: "low", "medium", or "high"

If no suitable data exists, respond with an empty list.

Return your answer as valid JSON:
{
  "data_sources": [
    {
      "source_category": "...",
      "source_description": "...",
      "availability": "...",
      "collection_difficulty": "..."
    }
  ]
}
"""

project_root = Path(getattr(CFG, "project_root", Path.cwd())).resolve()
raw_path = Path(CFG.prompt_data_map_path)
prompt_path = raw_path if raw_path.is_absolute() else (project_root / raw_path).resolve()

PROMPT_DATA_MAP = None

if prompt_path.exists():
    PROMPT_DATA_MAP = prompt_path.read_text(encoding="utf-8")
    print(f"Loaded data mapping prompt: {prompt_path}")
else:
    try:
        prompt_path.parent.mkdir(parents=True, exist_ok=True)
        prompt_path.write_text(PUBLIC_PLACEHOLDER_DATA_MAP_PROMPT, encoding="utf-8")
        print(f"Created placeholder prompt file (edit this for private version): {prompt_path}")
        PROMPT_DATA_MAP = prompt_path.read_text(encoding="utf-8")
        print(f"Loaded data mapping prompt: {prompt_path}")
    except Exception as e:
        PROMPT_DATA_MAP = PUBLIC_PLACEHOLDER_DATA_MAP_PROMPT
        warnings.warn(
            f"Prompt file not found and could not be created. Using public placeholder. "
            f"Looked for: {prompt_path}. Error: {e}",
            stacklevel=2
        )

print(f"Data mapping prompt ready ({len(PROMPT_DATA_MAP)} chars)")


# ------------------------------------------------------------
# Map gaps to data sources
# ------------------------------------------------------------
def map_gap_to_data(gap: Dict, max_retries: int = 3) -> List[Dict]:
    """Map a gap to potential data sources using LLM."""
    
    if CFG.dry_run:
        # Placeholder mapping
        return [
            {
                "source_category": "text",
                "source_description": "[DRY RUN] Placeholder data source",
                "availability": "public",
                "collection_difficulty": "medium"
            }
        ]
    
    # Format prompt (safe replacement: avoids .format() collisions with JSON braces)
    prompt = (
        PROMPT_DATA_MAP
        .replace("{gap_type}", str(gap.get("gap_type", "unknown")))
        .replace("{gap_statement}", str(gap.get("gap_statement", "")))
        .replace("{falsifiable_test}", str(gap.get("falsifiable_test", "")))
    )
    
    # Call LLM with retries
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=CFG.llm_model,
                messages=[
                    {"role": "system", "content": "You are a research data specialist. Always respond with valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=CFG.llm_temperature
            )
            
            content = response.choices[0].message.content.strip()
            
            # Handle markdown code blocks
            if content.startswith("```"):
                content = content.split("```")[1]
                if content.startswith("json"):
                    content = content[4:]
                content = content.strip()
            
            result = json.loads(content)
            
            # Validate structure
            if 'data_sources' not in result:
                raise ValueError("Response missing 'data_sources' key")
            
            return result['data_sources']
            
        except (json.JSONDecodeError, ValueError) as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries}: {e}")
            else:
                print(f"  Failed to parse JSON after {max_retries} attempts")
                return []
    
    return []

# ------------------------------------------------------------
# Update feasibility based on data availability
# ------------------------------------------------------------
def calculate_feasibility(data_sources: List[Dict]) -> float:
    """Calculate feasibility score from data sources."""
    if len(data_sources) == 0:
        return 0.1  # Low feasibility if no data available
    
    # Score each source
    source_scores = []
    for source in data_sources:
        availability = source.get('availability', 'unavailable')
        difficulty = source.get('collection_difficulty', 'high')
        
        # Availability score
        avail_score = {
            'public': 1.0,
            'restricted': 0.6,
            'unavailable': 0.1
        }.get(availability, 0.1)
        
        # Difficulty score
        diff_score = {
            'low': 1.0,
            'medium': 0.7,
            'high': 0.4
        }.get(difficulty, 0.4)
        
        source_scores.append(avail_score * diff_score)
    
    # Use max score (best available source)
    return max(source_scores)

# ------------------------------------------------------------
# Process all gaps
# ------------------------------------------------------------
print(f"\nMapping gaps to data sources...")
print(f"Dry run: {CFG.dry_run}")

for i, gap in enumerate(normalized_gaps, 1):
    if i % 10 == 0:
        print(f"Processing gap {i}/{len(normalized_gaps)}...")
    
    data_sources = map_gap_to_data(gap)
    gap['data_sources'] = data_sources
    
    # Update feasibility score
    gap['feasibility_score'] = calculate_feasibility(data_sources)
    
    # Recalculate priority score
    gap['priority_score'] = (
        CFG.priority_weights['importance'] * gap['importance_score'] +
        CFG.priority_weights['feasibility'] * gap['feasibility_score'] +
        CFG.priority_weights['novelty'] * gap['novelty_score']
    )

# Re-sort by updated priority
normalized_gaps.sort(key=lambda x: x['priority_score'], reverse=True)

print("Data mapping complete")
print(f"Gaps with data sources: {sum(1 for g in normalized_gaps if len(g.get('data_sources', [])) > 0)}")

# ------------------------------------------------------------
# Save updated gaps with data mappings
# ------------------------------------------------------------
gaps_with_data_path = Path(CFG.output_dir) / "gaps_with_data_mapping.json"
with open(gaps_with_data_path, 'w') as f:
    json.dump(normalized_gaps, f, indent=2)
print(f"Saved: {gaps_with_data_path}")

# Update CSV
gaps_ranked_df = pd.DataFrame(normalized_gaps)
for col in ['evidence_papers', 'cluster_id', 'merged_from', 'data_sources']:
    if col in gaps_ranked_df.columns:
        gaps_ranked_df[col] = gaps_ranked_df[col].apply(
            lambda x: '|'.join(map(str, x)) if isinstance(x, list) else str(x)
        )

ranked_path = Path(CFG.output_dir) / "gaps_ranked.csv"
gaps_ranked_df.to_csv(ranked_path, index=False)
print(f"Updated: {ranked_path}")

Created placeholder prompt file (edit this for private version): /Users/yuetoya/Desktop/researchOS100-private/notebooks/prompts/private/gap_to_data.txt
Loaded data mapping prompt: /Users/yuetoya/Desktop/researchOS100-private/notebooks/prompts/private/gap_to_data.txt
Data mapping prompt ready (1029 chars)

Mapping gaps to data sources...
Dry run: False
Processing gap 10/32...
Processing gap 20/32...
Processing gap 30/32...
Data mapping complete
Gaps with data sources: 32
Saved: artifacts/day22_gap_mining/20260118_142302/gaps_with_data_mapping.json
Updated: artifacts/day22_gap_mining/20260118_142302/gaps_ranked.csv


In [20]:
# ============================================================
# Cell 10 — Cluster-wise Gap Dashboard
# ============================================================
# Overview:
# - Generate a dashboard for each cluster showing:
#   - Top-N gaps by type
#   - Evidence paper list
#   - Suggested next actions
# - Save as Markdown (Notion-ready) and HTML
#
# Inputs / Outputs:
# - Inputs: normalized_gaps, representatives_df
# - Outputs: cluster_dashboards/ (one file per cluster)
#
# Notes:
# - Dashboard includes actionable next steps

# ------------------------------------------------------------
# Create dashboard directory
# ------------------------------------------------------------
dashboard_dir = Path(CFG.output_dir) / "cluster_dashboards"
dashboard_dir.mkdir(exist_ok=True)

# Use community_id as cluster key in these artifacts
REP_CLUSTER_COL = "community_id"
REP_PAPER_COL = "paper_id"
REP_TITLE_COL = "title"
REP_YEAR_COL = "publication_year"
REP_CIT_COL = "cited_by_count"
REP_RANK_COL = "pagerank"


# ------------------------------------------------------------
# Generate dashboard for each cluster
# ------------------------------------------------------------
def generate_cluster_dashboard(cluster_id: str, cluster_gaps: List[Dict]) -> str:
    """Generate Markdown dashboard for a cluster."""
    
    md = f"# Cluster {cluster_id} — Research Gap Dashboard\n\n"
    md += f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    
    # Summary stats
    md += f"## Summary\n\n"
    md += f"- Total gaps identified: {len(cluster_gaps)}\n"
    
    gap_types = {}
    for gap in cluster_gaps:
        gtype = gap.get('gap_type', 'unknown')
        gap_types[gtype] = gap_types.get(gtype, 0) + 1
    
    for gtype, count in sorted(gap_types.items()):
        md += f"  - {gtype}: {count}\n"
    
    md += f"\n"
    
    # Top gaps by type
    md += f"## Top Gaps by Type\n\n"
    
    for gap_type in ['coverage', 'measurement', 'causal']:
        type_gaps = [g for g in cluster_gaps if g.get('gap_type') == gap_type]
        
        if len(type_gaps) == 0:
            continue
        
        md += f"### {gap_type.capitalize()} Gaps\n\n"
        
        for i, gap in enumerate(type_gaps[:5], 1):
            md += f"**{i}. {gap['gap_statement']}**\n\n"
            md += f"- Gap ID: `{gap['gap_id']}`\n"
            md += f"- Priority score: {gap['priority_score']:.3f}\n"
            md += f"- Evidence papers: {len(gap.get('evidence_papers', []))}\n"
            
            if 'falsifiable_test' in gap:
                md += f"- Falsifiable test: {gap['falsifiable_test']}\n"
            
            if 'data_sources' in gap and len(gap['data_sources']) > 0:
                md += f"- Candidate data sources:\n"
                for ds in gap['data_sources'][:3]:
                    md += f"  - {ds['source_category']}: {ds['source_description']} ({ds['availability']})\n"
            
            md += f"\n"
    
    # Evidence papers
    md += f"## Representative Papers\n\n"
    
    # Collect all evidence paper IDs for this cluster
    all_evidence_ids = set()
    for gap in cluster_gaps:
        if 'evidence_papers' in gap:
            all_evidence_ids.update(gap['evidence_papers'])
    
    # Filter representative papers for this cluster
    mask = representatives_df[REP_PAPER_COL].isin(list(all_evidence_ids))
    mask = mask & (representatives_df[REP_CLUSTER_COL].astype(str) == str(cluster_id))
    
    cluster_reps = representatives_df[mask]

    
    if len(cluster_reps) > 0:
        for _, paper in cluster_reps.head(10).iterrows():
            title = paper.get('title', 'Unknown title')
            year = paper.get('publication_year', paper.get('year', 'N/A'))
            citations = paper.get('cited_by', 'N/A')
            
            md += f"- **{title}** ({year})\n"
            md += f"  - Citations: {citations}\n"
            
            if 'pagerank' in paper:
                md += f"  - PageRank: {paper['pagerank']:.4f}\n"
            
            md += f"\n"
    else:
        md += "No representative papers found.\n\n"
    
    # Suggested next actions
    md += f"## Suggested Next Actions\n\n"
    
    # Papers to read
    md += f"### Papers to Read\n\n"
    top_papers = cluster_reps.nlargest(5, 'rank') if 'rank' in cluster_reps.columns else cluster_reps.head(5)
    
    for _, paper in top_papers.iterrows():
        title = paper.get('title', 'Unknown title')
        md += f"- {title}\n"
    
    md += f"\n"
    
    # Data to collect
    md += f"### Data to Collect\n\n"
    all_data_sources = set()
    for gap in cluster_gaps[:10]:
        if 'data_sources' in gap:
            for ds in gap['data_sources']:
                all_data_sources.add(f"{ds['source_category']}: {ds['source_description']}")
    
    if len(all_data_sources) > 0:
        for ds in list(all_data_sources)[:10]:
            md += f"- {ds}\n"
    else:
        md += "No specific data sources identified.\n"
    
    md += f"\n"
    
    # Follow-up search queries
    md += f"### Follow-up Search Queries\n\n"
    
    for gap in cluster_gaps[:3]:
        gap_stmt = gap['gap_statement']
        # Extract key terms (simple heuristic)
        key_terms = [word for word in gap_stmt.split() if len(word) > 6][:3]
        if len(key_terms) > 0:
            query = " ".join(key_terms)
            md += f"- `{query}`\n"
    
    md += f"\n"
    
    return md

# ------------------------------------------------------------
# Generate dashboards for all clusters
# ------------------------------------------------------------
print(f"\nGenerating cluster dashboards...")

# Group gaps by cluster
cluster_gap_map = {}
for gap in normalized_gaps:
    clusters = gap.get('cluster_id', [])
    if not isinstance(clusters, list):
        clusters = [clusters]
    
    for cid in clusters:
        if cid not in cluster_gap_map:
            cluster_gap_map[cid] = []
        cluster_gap_map[cid].append(gap)

# Generate dashboard for each cluster
for cluster_id, cluster_gaps in cluster_gap_map.items():
    # Sort gaps by priority
    cluster_gaps.sort(key=lambda x: x['priority_score'], reverse=True)
    
    # Generate markdown
    md_content = generate_cluster_dashboard(cluster_id, cluster_gaps)
    
    # Save markdown
    md_path = dashboard_dir / f"cluster_{cluster_id}_dashboard.md"
    with open(md_path, 'w') as f:
        f.write(md_content)
    
    print(f"Generated dashboard for cluster {cluster_id}: {md_path}")

print(f"\nDashboards saved to {dashboard_dir}")


Generating cluster dashboards...
Generated dashboard for cluster 2: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_2_dashboard.md
Generated dashboard for cluster 0: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_0_dashboard.md
Generated dashboard for cluster 3: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_3_dashboard.md
Generated dashboard for cluster 1: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_1_dashboard.md
Generated dashboard for cluster 5: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_5_dashboard.md
Generated dashboard for cluster 4: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_4_dashboard.md
Generated dashboard for cluster 6: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_6_dashboard.md
Generated dashboard for cluster 7: artifacts/day22_gap_mining/20260118_142302/cluster_dashboards/cluster_7_dashboard.md

Dashb

In [30]:
# ============================================================
# Cell 11 — Notion Export (requests-based)
# - Find "Research Gaps" DB (or use NOTION_GAPS_DB_ID)
# - If missing, create it under NOTION_PARENT_PAGE_ID
# - Ensure required properties exist (adds them if missing)
# - Upsert normalized_gaps using the Title property (usually "Name") as key (= gap_id)
# ============================================================

import os
import re
import time
import requests
from datetime import datetime

# -----------------------------
# Config
# -----------------------------
DB_NAME = os.getenv("NOTION_GAPS_DB_NAME", "Research Gaps")
NOTION_TOKEN = os.getenv("NOTION_TOKEN")
NOTION_VERSION = os.getenv("NOTION_VERSION", "2022-06-28")

NOTION_GAPS_DB_ID_RAW = os.getenv("NOTION_GAPS_DB_ID")          # optional
NOTION_PARENT_PAGE_ID_RAW = os.getenv("NOTION_PARENT_PAGE_ID")  # required only if DB must be created

SLEEP_SEC = float(os.getenv("NOTION_SLEEP_SEC", "0.2"))

if not getattr(CFG, "enable_notion_export", False):
    print("\nNotion export disabled (CFG.enable_notion_export=False)")
    raise SystemExit

if NOTION_TOKEN is None:
    raise ValueError("NOTION_TOKEN could not be loaded from env")

NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

# -----------------------------
# Notion ID normalization
# -----------------------------
def normalize_notion_id(s: str) -> str:
    if not s:
        return None
    s = str(s).strip()

    m = re.search(r"([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})", s)
    if m:
        return m.group(1).lower()

    m = re.search(r"([0-9a-fA-F]{32})", s)
    if m:
        h = m.group(1).lower()
        return f"{h[0:8]}-{h[8:12]}-{h[12:16]}-{h[16:20]}-{h[20:32]}"

    return None

NOTION_GAPS_DB_ID = normalize_notion_id(NOTION_GAPS_DB_ID_RAW) if NOTION_GAPS_DB_ID_RAW else None
NOTION_PARENT_PAGE_ID = normalize_notion_id(NOTION_PARENT_PAGE_ID_RAW) if NOTION_PARENT_PAGE_ID_RAW else None

# -----------------------------
# HTTP helpers
# -----------------------------
def notion_get(url):
    return requests.get(url, headers=NOTION_HEADERS, timeout=60)

def notion_post(url, payload):
    return requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=60)

def notion_patch(url, payload):
    return requests.patch(url, headers=NOTION_HEADERS, json=payload, timeout=60)

# -----------------------------
# Auth check
# -----------------------------
r = notion_get("https://api.notion.com/v1/users/me")
if r.status_code != 200:
    print("❌ Notion authentication failed")
    print(r.json())
    raise RuntimeError("Notion auth failed")
print("✅ Notion authentication OK")

# -----------------------------
# Schema + mapping helpers
# -----------------------------
def chunk_rich_text(s: str, chunk_size: int = 1800):
    if s is None:
        s = ""
    s = str(s)
    if not s.strip():
        return []
    chunks = [s[i:i+chunk_size] for i in range(0, len(s), chunk_size)]
    return [{"type": "text", "text": {"content": c}} for c in chunks]

def safe_number(x):
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None

def normalize_clusters(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(v) for v in value]
    return [str(value)]

def get_db_schema(db_id: str) -> dict:
    resp = notion_get(f"https://api.notion.com/v1/databases/{db_id}")
    if resp.status_code != 200:
        raise RuntimeError(f"Notion get DB failed: {resp.status_code} {resp.text}")
    return resp.json()

def get_title_property_name(schema: dict) -> str:
    for name, meta in (schema.get("properties") or {}).items():
        if meta.get("type") == "title":
            return name
    # Practically should never happen; DB must have a title prop.
    raise RuntimeError("No title property found in DB schema")

def ensure_db_properties(db_id: str, schema: dict, required_props: dict) -> dict:
    """
    Ensure required properties exist in DB.
    Notion API allows adding properties via PATCH /v1/databases/{db_id}.
    """
    existing = set((schema.get("properties") or {}).keys())
    to_add = {k: v for k, v in required_props.items() if k not in existing}
    if not to_add:
        return schema

    payload = {"properties": to_add}
    resp = notion_patch(f"https://api.notion.com/v1/databases/{db_id}", payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Notion DB update (add properties) failed: {resp.status_code} {resp.text}")

    # Re-fetch schema
    return get_db_schema(db_id)

def gap_to_properties(gap: dict, title_prop: str) -> dict:
    gap_id = str(gap.get("gap_id", "")).strip()
    stmt = gap.get("gap_statement", "") or ""
    gtype = gap.get("gap_type", "unknown") or "unknown"
    clusters = normalize_clusters(gap.get("cluster_id", []))

    props = {
        # Key: store gap_id in the DB's title property (e.g., "Name")
        title_prop: {"title": [{"type": "text", "text": {"content": gap_id}}]},
        "Statement": {"rich_text": chunk_rich_text(stmt)},
        "Gap Type": {"select": {"name": str(gtype)}},
        "Falsifiable Test": {"rich_text": chunk_rich_text(gap.get("falsifiable_test", "") or "")},
        "Counterarguments": {"rich_text": chunk_rich_text(gap.get("counterarguments", "") or "")},
        "Cluster IDs": {"multi_select": [{"name": c} for c in clusters]},
        "Status": {"select": {"name": str(gap.get("status", "triage"))}},
        "Run Date": {"date": {"start": datetime.now().date().isoformat()}},
        "Last Pipeline Update": {"date": {"start": datetime.now().isoformat()}},
    }

    for k, v in {
        "Priority Score": gap.get("priority_score"),
        "Importance Score": gap.get("importance_score"),
        "Novelty Score": gap.get("novelty_score"),
        "Feasibility Score": gap.get("feasibility_score"),
    }.items():
        nv = safe_number(v)
        if nv is not None:
            props[k] = {"number": nv}

    return props

def filter_props_to_schema(props: dict, schema: dict) -> dict:
    existing = set((schema.get("properties") or {}).keys())
    return {k: v for k, v in props.items() if k in existing}

# -----------------------------
# Find or create DB
# -----------------------------
def find_database_id_by_name(db_name: str):
    cursor = None
    while True:
        payload = {
            "query": db_name,
            "filter": {"property": "object", "value": "database"},
            "page_size": 100,
        }
        if cursor:
            payload["start_cursor"] = cursor

        resp = notion_post("https://api.notion.com/v1/search", payload)
        if resp.status_code != 200:
            raise RuntimeError(f"Notion search failed: {resp.status_code} {resp.text}")

        data = resp.json()
        for db in data.get("results", []):
            title = "".join([t.get("plain_text", "") for t in db.get("title", [])]).strip()
            if title.lower() == db_name.lower():
                return db["id"]

        if not data.get("has_more"):
            break
        cursor = data.get("next_cursor")

    return None

def create_research_gaps_database(parent_page_id: str, db_name: str):
    payload = {
        "parent": {"type": "page_id", "page_id": parent_page_id},
        "title": [{"type": "text", "text": {"content": db_name}}],
        "properties": {
            "Name": {"title": {}},  # Title prop (Notion may still call it Name in UI)
        },
    }
    resp = notion_post("https://api.notion.com/v1/databases", payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Notion create DB failed: {resp.status_code} {resp.text}")
    return resp.json()["id"]

db_id = NOTION_GAPS_DB_ID

if not db_id:
    print(f'Searching for Notion database named "{DB_NAME}"...')
    db_id = find_database_id_by_name(DB_NAME)

if not db_id:
    if not NOTION_PARENT_PAGE_ID:
        raise ValueError(
            f'Could not find DB "{DB_NAME}" and NOTION_PARENT_PAGE_ID is not set or invalid. '
            f"Raw value was: {NOTION_PARENT_PAGE_ID_RAW}."
        )
    print(f'Creating Notion database "{DB_NAME}" under parent page UUID: {NOTION_PARENT_PAGE_ID}')
    db_id = create_research_gaps_database(NOTION_PARENT_PAGE_ID, DB_NAME)
    print(f'✅ Created DB "{DB_NAME}": {db_id}')
else:
    print(f'✅ Using DB "{DB_NAME}": {db_id}')

# -----------------------------
# Ensure properties exist (adds them if missing)
# -----------------------------
db_schema = get_db_schema(db_id)
TITLE_PROP = get_title_property_name(db_schema)
print("DB properties (before):", list(db_schema.get("properties", {}).keys()))
print("Using title property as key:", TITLE_PROP)

REQUIRED_PROPERTIES = {
    "Statement": {"rich_text": {}},
    "Gap Type": {"select": {"options": [{"name": "coverage"}, {"name": "measurement"}, {"name": "causal"}, {"name": "unknown"}]}},
    "Falsifiable Test": {"rich_text": {}},
    "Counterarguments": {"rich_text": {}},
    "Priority Score": {"number": {}},
    "Importance Score": {"number": {}},
    "Novelty Score": {"number": {}},
    "Feasibility Score": {"number": {}},
    "Cluster IDs": {"multi_select": {}},
    "Status": {"select": {"options": [{"name": "triage"}, {"name": "to_read"}, {"name": "designing_test"}, {"name": "data_collection"}, {"name": "in_progress"}, {"name": "done"}, {"name": "parked"}]}},
    "Run Date": {"date": {}},
    "Last Pipeline Update": {"date": {}},
}

db_schema = ensure_db_properties(db_id, db_schema, REQUIRED_PROPERTIES)
print("DB properties (after):", list(db_schema.get("properties", {}).keys()))

# -----------------------------
# Upsert pages by title (= gap_id)
# -----------------------------
def query_page_by_title(db_id: str, title_prop: str, title_value: str):
    payload = {"filter": {"property": title_prop, "title": {"equals": title_value}}, "page_size": 1}
    resp = notion_post(f"https://api.notion.com/v1/databases/{db_id}/query", payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Notion DB query failed: {resp.status_code} {resp.text}")
    results = resp.json().get("results", [])
    return results[0] if results else None

def create_page(db_id: str, props: dict):
    payload = {"parent": {"database_id": db_id}, "properties": props}
    resp = notion_post("https://api.notion.com/v1/pages", payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Notion page create failed: {resp.status_code} {resp.text}")
    return resp.json()["id"]

def update_page(page_id: str, props: dict):
    payload = {"properties": props}
    resp = notion_patch(f"https://api.notion.com/v1/pages/{page_id}", payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Notion page update failed: {resp.status_code} {resp.text}")
    return resp.json()["id"]

print(f"\nUpserting {len(normalized_gaps)} gaps to Notion...")

created_n = 0
updated_n = 0
failed_n = 0
total = len(normalized_gaps)

for i, gap in enumerate(normalized_gaps, 1):
    gap_id = str(gap.get("gap_id", "")).strip()
    if not gap_id:
        failed_n += 1
        continue

    try:
        existing = query_page_by_title(db_id, TITLE_PROP, gap_id)
        props = gap_to_properties(gap, TITLE_PROP)
        props = filter_props_to_schema(props, db_schema)

        if existing:
            update_page(existing["id"], props)
            updated_n += 1
        else:
            create_page(db_id, props)
            created_n += 1

    except Exception as e:
        failed_n += 1
        if failed_n <= 10:
            print(f"[Notion upsert failed] gap_id={gap_id} error={e}")

    if (i % 25) == 0 or i == total:
        print(f"Progress: {i}/{total} (created={created_n}, updated={updated_n}, failed={failed_n})")

    if SLEEP_SEC and SLEEP_SEC > 0:
        time.sleep(SLEEP_SEC)

print("\n✅ Notion upsert complete")
print(f"Created: {created_n}")
print(f"Updated: {updated_n}")
print(f"Failed:  {failed_n}")
print(f'Database: "{DB_NAME}" ({db_id})')


✅ Notion authentication OK
Searching for Notion database named "Research Gaps"...
✅ Using DB "Research Gaps": 2ec8e0e4-d162-803f-9044-ce1fc508585d
DB properties (before): ['Name']
Using title property as key: Name
DB properties (after): ['Importance Score', 'Feasibility Score', 'Priority Score', 'Status', 'Run Date', 'Counterarguments', 'Novelty Score', 'Last Pipeline Update', 'Gap Type', 'Statement', 'Falsifiable Test', 'Cluster IDs', 'Name']

Upserting 32 gaps to Notion...
Progress: 25/32 (created=25, updated=0, failed=0)
Progress: 32/32 (created=32, updated=0, failed=0)

✅ Notion upsert complete
Created: 32
Updated: 0
Failed:  0
Database: "Research Gaps" (2ec8e0e4-d162-803f-9044-ce1fc508585d)


In [31]:
# ============================================================
# Cell 12 — Diff vs Previous Run (minimal, Notion-based)
# ============================================================
# What it does:
# - Pull the current state from Notion "Research Gaps" DB (as the "previous run" store)
# - Compare with the current in-memory `normalized_gaps`
# - Report:
#   1) New gaps (in current run but not in Notion)
#   2) Missing gaps (in Notion but not in current run)  [optional signal]
#   3) Priority score changes above a threshold
# - Save a diff report JSON to CFG.output_dir
#
# Assumptions:
# - Your Notion DB uses the Title property (e.g., "Name") as the gap_id key (gap_cluster_X_YYY)
# - NOTION_TOKEN and NOTION_VERSION are set
# - NOTION_GAPS_DB_ID is set (recommended). If not, we search by name.
# - This is "minimal": no fancy matching beyond gap_id.

import os
import re
import json
import requests
from datetime import datetime
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
DB_NAME = os.getenv("NOTION_GAPS_DB_NAME", "Research Gaps")
NOTION_TOKEN = os.getenv("NOTION_TOKEN")
NOTION_VERSION = os.getenv("NOTION_VERSION", "2022-06-28")
NOTION_GAPS_DB_ID_RAW = os.getenv("NOTION_GAPS_DB_ID")  # recommended to set

PRIORITY_DELTA_THRESHOLD = float(getattr(CFG, "diff_priority_delta_threshold", 0.05))

if NOTION_TOKEN is None:
    raise ValueError("NOTION_TOKEN is not set")

NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

# -----------------------------
# Helpers
# -----------------------------
def normalize_notion_id(s: str) -> str:
    if not s:
        return None
    s = str(s).strip()

    m = re.search(r"([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})", s)
    if m:
        return m.group(1).lower()

    m = re.search(r"([0-9a-fA-F]{32})", s)
    if m:
        h = m.group(1).lower()
        return f"{h[0:8]}-{h[8:12]}-{h[12:16]}-{h[16:20]}-{h[20:32]}"

    return None

def notion_get(url):
    return requests.get(url, headers=NOTION_HEADERS, timeout=60)

def notion_post(url, payload):
    return requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=60)

def get_db_schema(db_id: str) -> dict:
    r = notion_get(f"https://api.notion.com/v1/databases/{db_id}")
    if r.status_code != 200:
        raise RuntimeError(f"Notion get DB failed: {r.status_code} {r.text}")
    return r.json()

def get_title_property_name(schema: dict) -> str:
    for name, meta in (schema.get("properties") or {}).items():
        if meta.get("type") == "title":
            return name
    raise RuntimeError("No title property found in DB schema")

def extract_title_text(page: dict, title_prop: str) -> str:
    prop = (page.get("properties") or {}).get(title_prop, {})
    items = prop.get("title", []) or []
    return "".join([t.get("plain_text", "") for t in items]).strip()

def extract_number(page: dict, prop_name: str):
    prop = (page.get("properties") or {}).get(prop_name, {})
    # number properties have {"type":"number","number": <val>}
    return prop.get("number", None)

def find_database_id_by_name(db_name: str):
    cursor = None
    while True:
        payload = {
            "query": db_name,
            "filter": {"property": "object", "value": "database"},
            "page_size": 100,
        }
        if cursor:
            payload["start_cursor"] = cursor

        r = notion_post("https://api.notion.com/v1/search", payload)
        if r.status_code != 200:
            raise RuntimeError(f"Notion search failed: {r.status_code} {r.text}")

        data = r.json()
        for db in data.get("results", []):
            title = "".join([t.get("plain_text", "") for t in db.get("title", [])]).strip()
            if title.lower() == db_name.lower():
                return db["id"]

        if not data.get("has_more"):
            break
        cursor = data.get("next_cursor")
    return None

def fetch_all_pages(db_id: str):
    pages = []
    cursor = None
    while True:
        payload = {"page_size": 100}
        if cursor:
            payload["start_cursor"] = cursor

        r = notion_post(f"https://api.notion.com/v1/databases/{db_id}/query", payload)
        if r.status_code != 200:
            raise RuntimeError(f"Notion DB query failed: {r.status_code} {r.text}")

        data = r.json()
        pages.extend(data.get("results", []))

        if not data.get("has_more"):
            break
        cursor = data.get("next_cursor")
    return pages

# -----------------------------
# Resolve DB ID + schema
# -----------------------------
db_id = normalize_notion_id(NOTION_GAPS_DB_ID_RAW) if NOTION_GAPS_DB_ID_RAW else None
if not db_id:
    db_id = find_database_id_by_name(DB_NAME)
if not db_id:
    raise RuntimeError(f'Could not resolve Notion DB. Set NOTION_GAPS_DB_ID or create DB named "{DB_NAME}".')

db_schema = get_db_schema(db_id)
TITLE_PROP = get_title_property_name(db_schema)

# -----------------------------
# Pull "previous" from Notion
# -----------------------------
notion_pages = fetch_all_pages(db_id)

prev = {}
for p in notion_pages:
    gap_id = extract_title_text(p, TITLE_PROP)
    if not gap_id:
        continue
    prev[gap_id] = {
        "page_id": p.get("id"),
        "priority_score": extract_number(p, "Priority Score"),
        "importance_score": extract_number(p, "Importance Score"),
        "novelty_score": extract_number(p, "Novelty Score"),
        "feasibility_score": extract_number(p, "Feasibility Score"),
        "last_pipeline_update": (p.get("properties", {}).get("Last Pipeline Update", {}) or {}).get("date", {}),
    }

# -----------------------------
# Current run (from notebook)
# -----------------------------
curr = {}
for g in normalized_gaps:
    gid = str(g.get("gap_id", "")).strip()
    if not gid:
        continue
    curr[gid] = {
        "priority_score": g.get("priority_score", None),
        "importance_score": g.get("importance_score", None),
        "novelty_score": g.get("novelty_score", None),
        "feasibility_score": g.get("feasibility_score", None),
        "gap_type": g.get("gap_type", None),
        "cluster_id": g.get("cluster_id", None),
    }

prev_ids = set(prev.keys())
curr_ids = set(curr.keys())

new_ids = sorted(list(curr_ids - prev_ids))
missing_ids = sorted(list(prev_ids - curr_ids))

changed = []
for gid in sorted(list(curr_ids & prev_ids)):
    p_prev = prev[gid].get("priority_score", None)
    p_curr = curr[gid].get("priority_score", None)
    if p_prev is None or p_curr is None:
        continue
    try:
        delta = float(p_curr) - float(p_prev)
    except Exception:
        continue
    if abs(delta) >= PRIORITY_DELTA_THRESHOLD:
        changed.append({
            "gap_id": gid,
            "priority_prev": p_prev,
            "priority_curr": p_curr,
            "priority_delta": delta,
            "gap_type": curr[gid].get("gap_type"),
            "cluster_id": curr[gid].get("cluster_id"),
            "notion_page_id": prev[gid].get("page_id"),
        })

changed_sorted = sorted(changed, key=lambda x: abs(x["priority_delta"]), reverse=True)

report = {
    "timestamp": datetime.now().isoformat(),
    "notion_db_id": db_id,
    "title_property": TITLE_PROP,
    "thresholds": {"priority_delta": PRIORITY_DELTA_THRESHOLD},
    "counts": {
        "prev_notion_pages": len(prev_ids),
        "curr_gaps": len(curr_ids),
        "new": len(new_ids),
        "missing": len(missing_ids),
        "changed_priority": len(changed_sorted),
    },
    "new_gap_ids": new_ids,
    "missing_gap_ids": missing_ids,
    "changed_priority": changed_sorted,
}

# -----------------------------
# Print summary
# -----------------------------
print("\n=== Diff vs Previous Run (Notion) ===")
print(f"Notion DB: {db_id} (title key: {TITLE_PROP})")
print(f"Prev (Notion): {len(prev_ids)} pages | Curr (run): {len(curr_ids)} gaps")
print(f"New gaps: {len(new_ids)} | Missing gaps: {len(missing_ids)} | Priority changes (|Δ|>={PRIORITY_DELTA_THRESHOLD}): {len(changed_sorted)}")

if new_ids:
    print("\nNew gaps (top 10):")
    for gid in new_ids[:10]:
        print(" -", gid)

if changed_sorted:
    print("\nLargest priority changes (top 10):")
    for x in changed_sorted[:10]:
        print(f" - {x['gap_id']}: {x['priority_prev']} -> {x['priority_curr']} (Δ={x['priority_delta']:+.4f})")

# -----------------------------
# Save report
# -----------------------------
out_dir = Path(getattr(CFG, "output_dir", "."))
out_dir.mkdir(parents=True, exist_ok=True)

diff_path = out_dir / "diff_report_vs_notion.json"
with open(diff_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"\nSaved diff report: {diff_path}")



=== Diff vs Previous Run (Notion) ===
Notion DB: 2ec8e0e4-d162-803f-9044-ce1fc508585d (title key: Name)
Prev (Notion): 32 pages | Curr (run): 32 gaps
New gaps: 0 | Missing gaps: 0 | Priority changes (|Δ|>=0.05): 0

Saved diff report: artifacts/day22_gap_mining/20260118_145443/diff_report_vs_notion.json
